[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/traderbrasil/livro-dinheiro/blob/v2.0/notebooks/cap16_dolar_diversificacao.ipynb)

# Capítulo 16: Investir no exterior

Laboratório do livro *Dinheiro: Querer, poder e gastar* (2ª edição), de Flávio Lemos, CMT, CFP®.

Você baixa o dólar oficial (PTAX) e compara uma carteira só de ações brasileiras com outra que coloca uma parte nos Estados Unidos, tudo em reais.

**Como usar:** clique em cada célula e aperte o botão ▶ (ou Shift + Enter), de cima para baixo. Na primeira vez, o Colab pede para você entrar com uma conta do Google. Os campos marcados com `# MUDE AQUI` podem ser alterados com os seus números.

*Conteúdo educacional. Não é recomendação de investimento. Resultados passados não garantem resultados futuros. Os dados são baixados no dia em que você roda o notebook; por isso, os seus números podem ser diferentes dos do livro, que foram conferidos em outubro de 2026.*

In [ ]:
%pip install -q yfinance
# Funções de apoio (não precisa mexer)
import io, time, math, datetime as dt
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

plt.rcParams.update({"figure.figsize": (10, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
CORES = {"azul": "#1F3864", "verde": "#2E7D32", "laranja": "#E07B00", "vermelho": "#C62828",
         "cinza": "#757575", "azul_claro": "#5B9BD5", "roxo": "#6A1B9A"}

def brl(x, casas=2):
    """Formata um número como R$ 1.234,56."""
    s = f"{x:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "R$ " + s

def pct(x, casas=2):
    return f"{x*100:,.{casas}f}%".replace(",", "X").replace(".", ",").replace("X", ".")

def eixo_reais(ax):
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: brl(v, 0)))

def sgs(codigo, inicio="01/01/2000", fim=None):
    """Baixa uma série do Sistema Gerenciador de Séries do Banco Central (SGS).
    Séries diárias são baixadas em blocos de 10 anos, o limite da API."""
    fim = fim or dt.date.today().strftime("%d/%m/%Y")
    d0 = dt.datetime.strptime(inicio, "%d/%m/%Y").date()
    d1 = dt.datetime.strptime(fim, "%d/%m/%Y").date()
    partes = []
    while d0 <= d1:
        d_fim = min(d1, d0.replace(year=d0.year + 9, month=12, day=31))
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
               f"?formato=json&dataInicial={d0:%d/%m/%Y}&dataFinal={d_fim:%d/%m/%Y}")
        for tentativa in range(4):
            try:
                r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
                r.raise_for_status()
                dados = r.json()
                break
            except Exception:
                time.sleep(3 * (tentativa + 1))
        else:
            raise RuntimeError(f"Não consegui baixar a série {codigo}. Tente de novo em alguns minutos.")
        if dados:
            partes.append(pd.DataFrame(dados))
        d0 = d_fim + dt.timedelta(days=1)
    df = pd.concat(partes, ignore_index=True)
    df["data"] = pd.to_datetime(df["data"], dayfirst=True)
    df["valor"] = pd.to_numeric(df["valor"])
    return df.set_index("data")["valor"].sort_index()


## 1. O dólar desde o Plano Real
Série 3698 do Banco Central: PTAX de venda, média do mês.

In [ ]:
usd = sgs(3698, inicio='01/07/1994')
ax = usd.plot(color=CORES['verde'], lw=2, title='Dólar comercial (PTAX venda, média mensal, R$)')
ax.set_xlabel(''); plt.show()
print('Último valor:', brl(usd.iloc[-1], 4))

## 2. Ibovespa x S&P 500 em reais

In [ ]:
import yfinance as yf
INICIO = '2000-01-01'   # MUDE AQUI
bx = yf.download(['^BVSP', '^GSPC'], start=INICIO, interval='1mo', auto_adjust=True, progress=False)['Close']
bx.index = bx.index.to_period('M').to_timestamp()
df = pd.concat({'ibov': bx['^BVSP'], 'sp': bx['^GSPC'], 'usd': usd}, axis=1).dropna()
df['sp_brl'] = df['sp'] * df['usd']
r = df[['ibov', 'sp_brl']].pct_change().dropna()
acum = 10_000 * (1 + r).cumprod()
ax = acum.rename(columns={'ibov':'Ibovespa','sp_brl':'S&P 500 em reais'}).plot(lw=2, color=[CORES['laranja'], CORES['azul']], title='R$ 10 mil, sem dividendos')
eixo_reais(ax); ax.set_xlabel(''); plt.show()
print('Correlação dos retornos mensais:', round(r.corr().iloc[0,1], 2))

## 3. Uma parte fora do país
Carteira rebalanceada todo mês de janeiro, com uma fatia em S&P 500 em reais.

In [ ]:
FATIA_EXTERIOR = 0.20   # MUDE AQUI
def carteira(r, w):
    v = 10_000; pesos = np.array([1 - w, w]); hist = []
    alocado = v * pesos
    for data, linha in r.iterrows():
        alocado = alocado * (1 + linha.values)
        if data.month == 1:
            alocado = alocado.sum() * pesos
        hist.append(alocado.sum())
    return pd.Series(hist, index=r.index)
c0 = carteira(r, 0.0); c1 = carteira(r, FATIA_EXTERIOR)
for nome, c in ((f'100% Brasil', c0), (f'{pct(FATIA_EXTERIOR,0)} no exterior', c1)):
    dd = (c / c.cummax() - 1).min()
    print(f'{nome}: final {brl(c.iloc[-1])}, maior queda {pct(dd)}')

O resultado depende muito do período escolhido. O ponto do capítulo 16 é outro: não depender de um só país e de uma só moeda. O custo de enviar dinheiro está na [calculadora de câmbio e remessa](https://www.traderbrasil.com/blog/calculadora-cambio-remessa-iof-vet.php).

---
**Hub do livro:** https://www.traderbrasil.com/livro-dinheiro/ · **Capítulo 16 no hub:** https://www.traderbrasil.com/livro-dinheiro/cap16/

Achou um erro? Escreva para contato@traderbrasil.com.